Step 1: Configuration

In [ ]:
import os
import yaml
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any

# Centralized configuration for the RAG system
@dataclass
class Config:
    # Directory paths for different components
    data_dir: str = "./data"                    # Raw data storage
    corpus_dir: str = "./corpus"               # Source documents
    eval_dir: str = "./eval_results"           # Evaluation outputs
    logs_dir: str = "./logs"                   # Application logs
    models_dir: str = "./models"               # Trained model weights
    vector_store_dir: str = "./vector_store"   # Vector index storage
    lora_dir: str = "./lora_adapter"           # LoRA adapter weights

    # Model configurations
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"  # Text-to-vector model
    llm_model_name: str = "TheBloke/Llama-2-7B-GGUF"               # Base LLM
    llm_model_file: str = "llama-2-7b.Q4_K_M.gguf"                 # Quantized version
    flant5_model: str = "google/flan-t5-small"                     # Auxiliary model

    # Text chunking for document processing
    chunk_size: int = 300          # Characters per chunk
    chunk_overlap: int = 50        # Overlap to preserve context between chunks

    # FAISS vector database files
    faiss_index_path: str = "./vector_store/faiss_index.bin"       # Binary index
    faiss_mapping_path: str = "./vector_store/index_mapping.json"  # ID-to-doc mapping

    # Similarity thresholds for retrieval filtering
    syllabus_threshold: float = 0.35       # Minimum relevance for syllabus content
    quick_answer_threshold: float = 0.5    # Minimum relevance for quick responses

    # LoRA fine-tuning parameters
    lora_r: int = 16                       # Rank of low-rank matrices
    lora_alpha: int = 32                   # Scaling factor (higher = stronger adaptation)
    lora_dropout: float = 0.1              # Regularization dropout rate
    lora_epochs: int = 5                   # Training epochs
    learning_rate: float = 5e-5            # Optimizer learning rate
    lora_target_modules: List[str] = field(default_factory=lambda: ["q", "v", "k", "o"])  # Which attention layers to adapt

    # Dataset sizes
    train_size: int = 100                  # Number of training examples
    eval_size: int = 50                    # Number of evaluation examples
    top_k_retrieval: int = 5               # Documents to retrieve per query

    # Query enhancement
    query_expansion: bool = True           # Enable synonym generation
    expansion_variants: int = 3            # Number of query variants to create

    # Training loop settings
    per_device_train_batch_size: int = 2   # Batch size per GPU/CPU
    gradient_accumulation_steps: int = 4   # Steps before updating weights (effective batch = 2*4 = 8)
    warmup_steps: int = 100                # Learning rate warmup steps
    logging_steps: int = 50                # Log metrics every N steps
    save_steps: int = 200                  # Save checkpoint every N steps
    save_total_limit: int = 3              # Keep only last N checkpoints

    # Logging configuration
    log_level: str = "INFO"                # Logging verbosity
    log_format: str = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

    def to_dict(self) -> Dict:   # Export configuration as dictionary for serialization.
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

    def save(self, path: str = "./config.yaml"):   # Save configuration to YAML file for reproducibility.
        with open(path, 'w') as f:
            yaml.dump(self.to_dict(), f, default_flow_style=False)
        print(f"Config saved to {path}")

# Initialize with default settings
config = Config()

# Ensure all required directories exist
for dir_path in [config.data_dir, config.corpus_dir, config.eval_dir,
                 config.logs_dir, config.models_dir, config.vector_store_dir,
                 config.lora_dir]:
    os.makedirs(dir_path, exist_ok=True)  # exist_ok=True prevents errors if already exists

# Persist configuration for tracking
config.save()

# Log current configuration summary
print("Configuration initialized")
print(f"Training with {config.train_size} QA pairs")
print(f"Testing with {config.eval_size} questions")
print(f"LoRA r={config.lora_r}, epochs={config.lora_epochs}")

Step 2: Logging Setup

In [ ]:
import logging
import sys
from logging.handlers import RotatingFileHandler

def setup_logging(config: Config):     # Configure logging with both console and rotating file outputs.

    logger = logging.getLogger('RAGSystem')   # Create main logger instance for the RAG system
    logger.setLevel(getattr(logging, config.log_level))  # Set log level from config (e.g., INFO, DEBUG)
    logger.handlers.clear()     # Remove any existing handlers to avoid duplicate logs

    # Console Handler (stdout)
    # For real-time visibility during development/running
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)  # Only show INFO+ in console

    # Simple console format with timestamp and level
    console_format = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(console_format)
    logger.addHandler(console_handler)

    # File Handler (rotating)
    # For persistent storage with rotation to prevent disk overflow
    file_handler = RotatingFileHandler(
        f"{config.logs_dir}/system.log",  # Log file path
        maxBytes=10*1024*1024,            # Rotate at 10 MB
        backupCount=5                     # Keep 5 backup files
    )
    file_handler.setLevel(logging.DEBUG)  # Capture everything in files

    # Use full format from config for detailed debugging
    file_format = logging.Formatter(config.log_format)
    file_handler.setFormatter(file_format)
    logger.addHandler(file_handler)

    return logger

# Initialize logger with configuration
logger = setup_logging(config)

# Log system startup
logger.info("RAG System Starting.")

Step 3: Installations

In [ ]:
# Install core dependencies for RAG system
!pip install -q \
    llama-cpp-python \
    haystack-ai \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    rouge-score \
    nltk \
    pandas \
    numpy \
    tqdm \
    scikit-learn \
    torch \
    huggingface_hub \
    peft \
    accelerate \
    transformers \
    requests \
    beautifulsoup4 \
    pyyaml

# Upgrade PyTorch AO (Advanced Optimizations) for better performance
!pip install -q --upgrade torchao>=0.16.0a

# Log success message after installation
logger.info("All installations completed")

Step 4: Imports

In [ ]:
# Suppress warning messages for cleaner output
import warnings
warnings.filterwarnings("ignore")

# Standard library imports
import json                         # JSON file handling
import re                           # Regular expressions for text processing
import pickle                       # Serialize Python objects
from datetime import datetime       # Timestamps and logging
from typing import List, Dict, Tuple, Optional, Any  # Type hints
import requests                     # HTTP requests for APIs
import zipfile                      # Handle zip archives
import faiss                        # Facebook AI Similarity Search (vector DB)
import fitz                         # PyMuPDF - PDF processing
import pandas as pd                 # Data manipulation
import numpy as np                  # Numerical operations
from tqdm import tqdm               # Progress bars
import time                         # Sleep/delay functions
import nltk                         # Natural Language Toolkit
nltk.download("punkt", quiet=True)      # Tokenizer for sentence splitting
nltk.download("punkt_tab", quiet=True)  # Tabular tokenizer support

# PyTorch imports
import torch                        # Deep learning framework
from torch.utils.data import Dataset, DataLoader  # Data handling utilities

# LLM inference
from llama_cpp import Llama         # Run GGUF models locally
from huggingface_hub import hf_hub_download  # Download models from HF Hub

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,                  # Load tokenizers
    AutoModelForSeq2SeqLM,          # Sequence-to-sequence models (Flan-T5)
    TrainingArguments,              # Training configuration
    Trainer,                        # Training loop
    DataCollatorForSeq2Seq          # Batch collation for Seq2Seq
)

# LoRA for parameter-efficient fine-tuning
from peft import (
    LoraConfig,                     # LoRA configuration
    get_peft_model,                 # Apply LoRA to model
    TaskType,                       # Task type
    PeftModel                       # Load pre-trained LoRA adapters
)

# Haystack RAG framework components
from haystack import Document                    # Document representation
from haystack.document_stores.in_memory import InMemoryDocumentStore  # Vector storage
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,  # Embed documents
    SentenceTransformersTextEmbedder       # Embed queries
)
from haystack.components.retrievers import InMemoryEmbeddingRetriever  # Similarity search

# Evaluation metrics
from rouge_score import rouge_scorer       # ROUGE scores (text quality)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction  # BLEU scores

# Machine learning utilities
from sklearn.metrics.pairwise import cosine_similarity  # Similarity computation
from sklearn.cluster import KMeans                      # Clustering for query expansion

# Determine compute device (GPU if available, fallback to CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Device: {DEVICE}")

Step 5: Data Loader

In [ ]:
class PDFDownloader:   # Handles PDF downloading from NCERT website
    def __init__(self, config: Config, logger: logging.Logger):  # Initialize with configuration and logger instance.

        self.config = config
        self.logger = logger

    def download_chapter(self, ch_num: int) -> bool:    # Download a single chapter PDF. Returns True if successful.

        ch_num_str = str(ch_num).zfill(2)     # Format chapter number as 2-digit string (e.g., 1 -> "01")
        url = f"https://ncert.nic.in/textbook/pdf/hesc1{ch_num_str}.pdf"  # NCERT URL pattern for chapter PDFs
        save_path = f"{self.config.data_dir}/chapter_{ch_num}.pdf"

        # Check if file exists and is valid (starts with PDF header)
        if os.path.exists(save_path):
            with open(save_path, 'rb') as f:
                if f.read(4) == b'%PDF':
                    return True  # Already downloaded and valid

        # Retry up to 3 times with exponential backoff
        for attempt in range(3):
            try:
                # Mimic a browser to avoid blocking
                headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
                response = requests.get(url, headers=headers, timeout=30)

                # Verify successful download and valid PDF content
                if response.status_code == 200 and response.content[:4] == b'%PDF':
                    with open(save_path, "wb") as f:
                        f.write(response.content)
                    return True

                # Exponential backoff before retry: 1s, 2s, 4s
                time.sleep(2 ** attempt)
            except:
                time.sleep(2 ** attempt)
        return False

    def download_all(self):   # Download all 13 chapters. Returns summary of successes and failures.

        results = {"success": [], "failed": []}
        for ch_num in range(1, 14):  # Chapters 1 through 13
            if self.download_chapter(ch_num):
                results["success"].append(ch_num)
            else:
                results["failed"].append(ch_num)
        return results

class PDFExtractor:   # Extracts and cleans text from PDFs

    # Handles text extraction and cleaning from downloaded PDFs
    def __init__(self, config: Config, logger: logging.Logger):   # Initialize with configuration and logger.

        self.config = config
        self.logger = logger

    def extract_text(self, pdf_path: str) -> str:   # Extract and clean text from a single PDF file.

        try:
            # Open PDF using PyMuPDF
            doc = fitz.open(pdf_path)
            text = ""

            # Iterate through all pages
            for page_num in range(len(doc)):
                page = doc[page_num]
                page_text = page.get_text()

                # Clean page text:
                page_text = re.sub(r'Fig\.\s*\d+\.\d+', '', page_text)  # Remove figure references
                page_text = re.sub(r'Table\s*\d+\.\d+', '', page_text)  # Remove table references
                page_text = re.sub(r'\n\d+\n', '\n', page_text)          # Remove standalone page numbers
                text += page_text + "\n"

            doc.close()

            # Global text cleaning:
            text = re.sub(r'\s+', ' ', text)                    # Normalize whitespace
            text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text)     # Fix hyphenated line breaks
            text = re.sub(r'[^\x00-\x7F]+', '', text)           # Remove non-ASCII characters

            return text.strip()

        except Exception as e:
            self.logger.error(f"Error extracting text: {e}")
            return ""

    def extract_all(self) -> Dict[int, str]:   # Extract text from all chapter PDFs.
        chapter_texts = {}
        for ch_num in range(1, 14):
            pdf_path = f"{self.config.data_dir}/chapter_{ch_num}.pdf"

            # Process only if PDF exists
            if os.path.exists(pdf_path):
                text = self.extract_text(pdf_path)

                # Only keep if substantial text extracted (>500 chars)
                if len(text) > 500:
                    chapter_texts[ch_num] = text

        return chapter_texts

Step 6: Vector Store

In [ ]:
class VectorStore:   # Handles FAISS vector store operations

    # Manages document embedding, indexing, and similarity search using FAISS
    def __init__(self, config: Config, logger: logging.Logger):   # Initialize vector store with configuration and logger.
        self.config = config
        self.logger = logger
        self.embedder = None      # Sentence transformer model for embeddings
        self.index = None         # FAISS index for similarity search
        self.documents = []       # Original document objects
        self.mapping = []         # Metadata mapping for retrieved documents

    def initialize_embedder(self):   # Load the embedding model for text vectorization.
        from sentence_transformers import SentenceTransformer
        self.embedder = SentenceTransformer(self.config.embedding_model)
        return self.embedder

    def create_chunks(self, chapter_texts: Dict[int, str]) -> List[Document]:
        # Split chapter text into overlapping chunks for indexing.
        documents = []

        for ch_num, content in chapter_texts.items():
            # Split content into words for precise chunking
            words = content.split()

            # Create chunks with overlap to preserve context
            for i in range(0, len(words), self.config.chunk_size - self.config.chunk_overlap):
                chunk = ' '.join(words[i:i + self.config.chunk_size])

                # Only keep chunks with meaningful content
                if len(chunk) > 100:
                    doc = Document(
                        content=chunk,
                        meta={"chapter": ch_num, "title": self.get_chapter_title(ch_num)}
                    )
                    documents.append(doc)

        return documents

    def get_chapter_title(self, ch_num: int) -> str:  # Map chapter number to its title from NCERT Science textbook.

        titles = {
            1: "Crop Production and Management",
            2: "Microorganisms: Friend and Foe",
            3: "Coal and Petroleum",
            4: "Combustion and Flame",
            5: "Conservation of Plants and Animals",
            6: "Reproduction in Animals",
            7: "Reaching the Age of Adolescence",
            8: "Force and Pressure",
            9: "Friction",
            10: "Sound",
            11: "Chemical Effects of Electric Current",
            12: "Some Natural Phenomena",
            13: "Light"
        }
        return titles.get(ch_num, f"Chapter {ch_num}")

    def build_index(self, documents: List[Document]) -> bool:  # Build FAISS index from documents and save to disk.

        try:
            # Ensure embedder is loaded
            if self.embedder is None:
                self.initialize_embedder()

            # Generate embeddings for all documents
            texts = [doc.content for doc in documents]
            embeddings = self.embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)

            # Create FAISS index with L2 (Euclidean) distance
            self.index = faiss.IndexFlatL2(embeddings.shape[1])
            self.index.add(embeddings.astype('float32'))

            # Save index to disk
            faiss.write_index(self.index, self.config.faiss_index_path)

            # Save metadata mapping for retrieval
            self.mapping = []
            for i, doc in enumerate(documents):
                self.mapping.append({
                    "id": i,
                    "content": doc.content,
                    "chapter": doc.meta["chapter"],
                    "title": doc.meta["title"],
                    "content_preview": doc.content[:200] + "..."  # Truncated for display
                })

            # Save mapping as JSON for easy inspection
            with open(self.config.faiss_mapping_path, "w", encoding='utf-8') as f:
                json.dump(self.mapping, f, indent=2, ensure_ascii=False)

            self.documents = documents
            return True

        except Exception as e:
            self.logger.error(f"Error building FAISS index: {e}")
            return False

    def load_index(self) -> bool:
        """Load existing FAISS index and mapping from disk."""
        try:
            # Check if index exists
            if not os.path.exists(self.config.faiss_index_path):
                return False

            # Load FAISS index
            self.index = faiss.read_index(self.config.faiss_index_path)

            # Load metadata mapping
            with open(self.config.faiss_mapping_path, "r", encoding='utf-8') as f:
                self.mapping = json.load(f)

            return True

        except Exception as e:
            self.logger.error(f"Error loading FAISS index: {e}")
            return False

    def search(self, query: str, top_k: int = 5) -> List[Dict]:  # Search for most relevant documents using semantic similarity.

        # Ensure index and embedder are ready
        if self.index is None:
            return []
        if self.embedder is None:
            self.initialize_embedder()

        # Encode query to vector
        query_embedding = self.embedder.encode([query], convert_to_numpy=True)

        # Search FAISS index
        distances, indices = self.index.search(query_embedding.astype('float32'), top_k)

        # Format results with similarity scores
        results = []
        for i, idx in enumerate(indices[0]):
            if idx < len(self.mapping):
                doc = self.mapping[idx]
                # Convert L2 distance to similarity score (0 to 1 range)
                similarity_score = float(1 / (1 + distances[0][i]))
                results.append({
                    "content": doc["content"],
                    "chapter": doc["chapter"],
                    "title": doc["title"],
                    "score": similarity_score
                })

        return results

Step 7: Training Data Generator

In [ ]:
class TrainingDataGenerator:   # Generates enhanced training data from textbook content
    # Creates QA pairs from textbook chapters for fine-tuning
    def __init__(self, config: Config, logger: logging.Logger):   # Initialize with configuration and logger.
        self.config = config
        self.logger = logger
        # Pre-load question templates for each chapter
        self.question_templates = self._create_question_templates()

    def _create_question_templates(self) -> Dict[str, List[str]]:    # Define chapter-specific question templates for varied question generation.
        templates = {
            # Each chapter has 3 template styles tailored to its content
            1: ["What is {concept}?", "Explain the process of {concept}.", "How does {concept} help in agriculture?"],
            2: ["What are {concept}?", "Describe the role of {concept}.", "How do {concept} affect human health?"],
            3: ["What are {concept}?", "How are {concept} formed?", "Explain the uses of {concept}."],
            4: ["What is {concept}?", "Explain the process of {concept}.", "What are the types of {concept}?"],
            5: ["What is {concept}?", "Why is {concept} important?", "Explain the ways to {concept}."],
            6: ["What is {concept}?", "Explain the process of {concept}.", "What are the types of {concept}?"],
            7: ["What is {concept}?", "Explain the changes during {concept}.", "What are the health aspects of {concept}?"],
            8: ["What is {concept}?", "Explain the concept of {concept}.", "What are the types of {concept}?"],
            9: ["What is {concept}?", "Explain the types of {concept}.", "How is {concept} useful?"],
            10: ["What is {concept}?", "How is {concept} produced?", "What are the characteristics of {concept}?"],
            11: ["What is {concept}?", "Explain the {concept} of electricity.", "What are conductors and insulators?"],
            12: ["What is {concept}?", "Explain how {concept} occurs.", "What are the effects of {concept}?"],
            13: ["What is {concept}?", "Explain the {concept} of light.", "What are the properties of {concept}?"]
        }
        return templates

    def extract_concepts(self, chapter_text: str) -> List[str]:   # Extract key concepts from chapter text using pattern matching.
        concepts = []

        # Split text into sentences
        sentences = re.split(r'[.!?]+', chapter_text)

        # Look for capitalized multi-word phrases (likely concepts/terms)
        for sentence in sentences:
            # Pattern: Two or more capitalized words (e.g., "Crop Production", "Nitrogen Fixation")
            terms = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b', sentence)

            # Filter out common words and short terms
            for term in terms:
                if len(term) > 3 and term.lower() not in ['The', 'This', 'These', 'Those']:
                    concepts.append(term)

        # Remove duplicates and limit to top 20 concepts
        concepts = list(set(concepts))
        return concepts[:20]

    def generate_qa_pairs(self, chapter_texts: Dict[int, str]) -> List[Dict]:  # Generate question-answer pairs from all chapters.
        qa_pairs = []

        for ch_num, text in chapter_texts.items():
            # Extract key concepts from this chapter
            concepts = self.extract_concepts(text)

            # Get templates for this chapter (fallback to generic if not found)
            templates = self.question_templates.get(ch_num, ["What is {concept}?"])

            # Generate questions for top 5 concepts
            for concept in concepts[:5]:
                # Use up to 3 templates per concept
                for template in templates[:3]:
                    # Replace placeholder with actual concept
                    question = template.format(concept=concept)

                    # Find answer context from the text
                    answer = self._extract_answer(text, concept)

                    # Only keep QA pairs with meaningful answers
                    if answer and len(answer) > 20:
                        qa_pairs.append({
                            "input": question,
                            "output": answer,
                            "chapter": ch_num,
                            "concept": concept
                        })

        return qa_pairs

    def _extract_answer(self, text: str, concept: str) -> str:   # Extract sentences containing the concept to form the answer.
        # Split text into sentences
        sentences = re.split(r'[.!?]+', text)

        # Find sentences that contain the concept
        relevant_sentences = []
        for sentence in sentences:
            # Case-insensitive match and ensure meaningful content
            if concept.lower() in sentence.lower() and len(sentence.strip()) > 20:
                relevant_sentences.append(sentence.strip())

        # Return up to 3 relevant sentences as the answer
        if relevant_sentences:
            return '. '.join(relevant_sentences[:3]) + '.'

        return ""

Step 8: Download And Process PDFs

In [ ]:
# Log the start of PDF download process
logger.info("Downloading NCERT Class 8 Science PDFs.")

# Initialize PDF downloader with configuration and logger
downloader = PDFDownloader(config, logger)

# Download all 13 chapters from NCERT website
download_results = downloader.download_all()

# Log download summary showing successful downloads
logger.info(f"Downloaded: {len(download_results['success'])} chapters")

# Log the start of text extraction process
logger.info("Extracting text from PDFs.")

# Initialize PDF extractor for text extraction and cleaning
extractor = PDFExtractor(config, logger)

# Extract and clean text from all downloaded PDFs
# Returns dictionary: {chapter_number: extracted_text}
chapter_texts = extractor.extract_all()

# Log extraction summary showing how many chapters were successfully processed
logger.info(f"Extracted {len(chapter_texts)} chapters")

Step 9: Generate Enhanced Training Data

In [ ]:
# Log the start of training data generation
logger.info("Generating enhanced training data.")

# Initialize training data generator with configuration and logger
data_generator = TrainingDataGenerator(config, logger)

# Generate QA pairs from chapter texts
training_qa_pairs = data_generator.generate_qa_pairs(chapter_texts)

# Save generated training data to JSON file for persistence and reuse
with open(f"{config.corpus_dir}/enhanced_training_data.json", "w") as f:
    json.dump(training_qa_pairs, f, indent=2)

# Log total number of QA pairs generated
logger.info(f"Generated {len(training_qa_pairs)} training QA pairs")

# Sample data for training
# Set random seed for reproducibility
import random
random.seed(42)

# Sample training pairs to match configured training size
if len(training_qa_pairs) > config.train_size:
    # Randomly select subset (without replacement)
    training_pairs = random.sample(training_qa_pairs, config.train_size)
else:
    # Use all if fewer than configured size
    training_pairs = training_qa_pairs

# Log final training set size
logger.info(f"Using {len(training_pairs)} QA pairs for LoRA training")

Step 10: Build Vector Store

In [ ]:
# Log the start of FAISS vector store building
logger.info("Building FAISS Vector Store.")

# Initialize vector store with configuration and logger
vector_store = VectorStore(config, logger)

# Split chapter texts into overlapping chunks for better retrieval
documents = vector_store.create_chunks(chapter_texts)

# Load the embedding model (sentence transformer)
vector_store.initialize_embedder()

# Generate embeddings and build FAISS index
vector_store.build_index(documents)

# Log the number of chunks indexed
logger.info(f"Built FAISS index with {len(documents)} chunks")

# Save documents in JSONL format for easy ingestion by other applications
import json
import os
# Create corpus directory if it doesn't exist
os.makedirs("./corpus", exist_ok=True)

# Convert Haystack Document objects to dictionaries
documents_data = []
for doc in documents:
    documents_data.append({
        "content": doc.content,                # The actual text chunk
        "chapter": doc.meta["chapter"],        # Chapter number
        "title": doc.meta["title"]             # Chapter title
    })

# Save as JSONL (JSON Lines) - each line is a separate JSON object
# Useful for streaming and line-by-line processing
with open("./corpus/class8_science.jsonl", "w", encoding='utf-8') as f:
    for doc in documents_data:
        f.write(json.dumps(doc) + "\n")        # One document per line

# Save as JSON - for app.py compatibility
# Easier for humans to read and debug
with open("./corpus/documents.json", "w", encoding='utf-8') as f:
    json.dump(documents_data, f, indent=2, ensure_ascii=False)

# Log completion with document count
print(f"Saved {len(documents_data)} documents to ./corpus/")

Step 11: Enhanced LoRA Training

In [ ]:
# Log the start of LoRA fine-tuning
logger.info("Training Enhanced LoRA with 100 QA pairs.")

# Load Base Model
# Load FLAN-T5 tokenizer for text processing
flant5_tokenizer = AutoTokenizer.from_pretrained(config.flant5_model)
# Load base FLAN-T5 model and move to CPU (will be moved to GPU if available later)
flant5_base = AutoModelForSeq2SeqLM.from_pretrained(config.flant5_model).to("cpu")

# Configure LoRA
# Enhanced LoRA config with parameters from config
lora_config = LoraConfig(
    r=config.lora_r,                      # Rank of low-rank matrices
    lora_alpha=config.lora_alpha,         # Scaling factor
    target_modules=config.lora_target_modules,  # Which attention layers to adapt
    lora_dropout=config.lora_dropout,     # Dropout for regularization
    bias="none",                          # Don't train bias terms
    task_type=TaskType.SEQ_2_SEQ_LM,      # Sequence-to-sequence task
)

# Apply LoRA to base model (freezes base weights, adds trainable adapters)
flant5_model = get_peft_model(flant5_base, lora_config)
# Log number of trainable parameters
logger.info(f"Trainable params: {flant5_model.num_parameters(only_trainable=True):,}")

# Dataset Class
class EnhancedDataset(Dataset):   # PyTorch Dataset for QA pairs with concept enrichment.
    def __init__(self, pairs, tokenizer):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = 512  # Max tokens for input

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]

        # Format input with concept if available
        if p.get("concept"):
            text = f"Concept: {p['concept']}\nQuestion: {p['input']}\nAnswer:"
        else:
            text = f"Question: {p['input']}\nAnswer:"

        # Tokenize input (question + concept)
        inp = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        # Tokenize output (answer)
        out = self.tokenizer(
            p['output'],
            max_length=150,                # Answers are typically shorter
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": inp["input_ids"].squeeze(),        # Token IDs for input
            "attention_mask": inp["attention_mask"].squeeze(),  # Mask for padding
            "labels": out["input_ids"].squeeze(),           # Token IDs for output
        }

# Create dataset from training pairs
dataset = EnhancedDataset(training_pairs, flant5_tokenizer)

# Training Configuration
# Training arguments with parameters from config
training_args = TrainingArguments(
    output_dir=config.lora_dir,                    # Where to save checkpoints
    num_train_epochs=config.lora_epochs,           # Number of training epochs
    per_device_train_batch_size=config.per_device_train_batch_size,  # Batch size
    gradient_accumulation_steps=config.gradient_accumulation_steps,  # Simulate larger batches
    learning_rate=config.learning_rate,            # Optimizer learning rate
    warmup_steps=config.warmup_steps,              # LR warmup for stability
    logging_steps=config.logging_steps,            # Log metrics every N steps
    save_steps=config.save_steps,                  # Save checkpoint every N steps
    eval_strategy="no",                            # No evaluation during training
    report_to="none",                              # Disable external logging
    save_total_limit=config.save_total_limit,      # Keep only last N checkpoints
    load_best_model_at_end=False,                  # Don't auto-load best checkpoint
    fp16=torch.cuda.is_available(),                # Mixed precision if GPU available
)

# Create Hugging Face Trainer
trainer = Trainer(
    model=flant5_model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(flant5_tokenizer, model=flant5_model),
)

# Start Training
logger.info("Starting Enhanced FLAN-T5 LoRA Training.")
logger.info(f"   • Training samples: {len(dataset)}")
logger.info(f"   • Epochs: {config.lora_epochs}")
logger.info(f"   • Batch size: {config.per_device_train_batch_size}")
logger.info(f"   • Learning rate: {config.learning_rate}")
logger.info(f"   • LoRA r: {config.lora_r}")

# Run the training loop
trainer.train()
logger.info("Training complete!")

# Save Model
# Create directory for final model
os.makedirs(f"{config.lora_dir}/final_model", exist_ok=True)

# Save LoRA adapter weights
flant5_model.save_pretrained(f"{config.lora_dir}/final_model")
# Save tokenizer for future use
flant5_tokenizer.save_pretrained(f"{config.lora_dir}/final_model")
logger.info(f"Enhanced FLAN-T5 LoRA model saved to: {config.lora_dir}/final_model")

# Save Training Metadata
# Record training details for reproducibility
training_metadata = {
    "timestamp": datetime.now().isoformat(),       # When training completed
    "train_size": len(training_pairs),             # Number of samples used
    "epochs": config.lora_epochs,
    "learning_rate": config.learning_rate,
    "lora_r": config.lora_r,
    "lora_alpha": config.lora_alpha,
    "target_modules": config.lora_target_modules,
    "trainable_params": flant5_model.num_parameters(only_trainable=True),  # Size of adapters
}

# Save metadata as JSON
with open(f"{config.lora_dir}/training_metadata.json", "w") as f:
    json.dump(training_metadata, f, indent=2)

logger.info("Training metadata saved")

Step 12: Query Expansion

In [ ]:
import re
from typing import List, Dict

class QueryExpander:   #  Expands queries using synonyms and paraphrasing
    # Generates query variants to improve retrieval recall
    def __init__(self, config: Config, logger: logging.Logger):  # Initialize with configuration and logger.
        self.config = config
        self.logger = logger
        # Load synonym dictionary for concept expansion
        self.synonyms = self._load_synonyms()

    def _load_synonyms(self) -> Dict[str, List[str]]:    # Define synonyms for key scientific concepts in the textbook.
        synonyms = {
            # Physics concepts
            'force': ['push', 'pull', 'interaction', 'strength'],
            'friction': ['resistance', 'opposition', 'drag'],
            'sound': ['noise', 'vibration', 'wave'],
            'light': ['illumination', 'radiation', 'photon'],
            'pressure': ['force', 'stress', 'thrust'],
            'motion': ['movement', 'speed', 'velocity'],
            'gravity': ['attraction', 'weight', 'pull'],
            'temperature': ['heat', 'warmth', 'cold'],
            'electric': ['electrical', 'current', 'power'],
            'magnetic': ['magnet', 'attraction', 'repulsion'],
            'energy': ['power', 'force', 'strength'],

            # Biology concepts
            'crop': ['plant', 'agriculture', 'cultivation'],
            'microorganism': ['microbe', 'bacteria', 'germ', 'virus'],
            'reproduction': ['breeding', 'procreation', 'propagation'],
            'conservation': ['protection', 'preservation', 'safeguarding'],
            'photosynthesis': ['plant food', 'sunlight process'],
            'ecosystem': ['environment', 'habitat', 'biome'],
            'pollution': ['contamination', 'waste', 'toxins'],
            'climate': ['weather', 'temperature', 'atmosphere'],

            # Chemistry concepts
            'combustion': ['burning', 'fire', 'ignition'],
        }
        return synonyms

    def expand_query(self, query: str) -> List[str]:   # Generate multiple query variants for better retrieval.

        # Start with original query
        expansions = [query]
        query_lower = query.lower()

        # Strategy 1: Synonym Replacement
        # Replace words with their synonyms
        words = query_lower.split()
        for word in words:
            # Remove punctuation for matching
            word = word.strip('.,!?')

            if word in self.synonyms:
                # Use up to 2 synonyms per word
                for syn in self.synonyms[word][:2]:
                    # Preserve original capitalization
                    if word[0].isupper():
                        syn = syn.capitalize()
                    expanded = query.replace(word, syn)
                    if expanded not in expansions:
                        expansions.append(expanded)

        # Strategy 2: Paraphrasing
        # Add grammatical variations of common question forms
        if "what is" in query_lower:
            expansions.append(query.replace("What is", "Define"))
            expansions.append(query.replace("What is", "Explain"))

        if "how does" in query_lower:
            expansions.append(query.replace("How does", "Explain how"))

        if "why does" in query_lower:
            expansions.append(query.replace("Why does", "Why"))

        if "what are" in query_lower:
            expansions.append(query.replace("What are", "Define"))
            expansions.append(query.replace("What are", "List"))

        # Clean up
        # Remove duplicates
        expansions = list(set(expansions))

        # Ensure original query is first
        if query in expansions:
            expansions.remove(query)
            expansions.insert(0, query)

        # Limit number of variants (from config)
        return expansions[:self.config.expansion_variants]

# Initialize query expander
query_expander = QueryExpander(config, logger)
logger.info("Query Expander initialized")

Step 13: Answer Verification

In [ ]:
class AnswerVerifier:  # Verifies answer grounding and detects hallucinations

    # Checks if generated answers are grounded in retrieved context
    def __init__(self, vector_store: VectorStore, logger: logging.Logger):  # Initialize with vector store for embeddings.
        self.vector_store = vector_store
        self.logger = logger
        self.similarity_threshold = 0.4  # Minimum score to consider answer grounded

    def verify_grounding(self, answer: str, context: str) -> Dict[str, Any]:  # Check if answer is semantically grounded in the context.
        # Early validation for empty inputs
        if not answer or not context:
            return {
                "is_grounded": False,
                "score": 0.0,
                "reason": "Empty answer or context"
            }

        # Ensure embedder is loaded
        if self.vector_store.embedder is None:
            self.vector_store.initialize_embedder()

        try:
            # Method 1: Semantic Similarity
            # Compare meaning using embedding vectors
            answer_emb = self.vector_store.embedder.encode([answer], convert_to_numpy=True)
            context_emb = self.vector_store.embedder.encode([context], convert_to_numpy=True)

            # Cosine similarity: measures semantic closeness
            similarity = cosine_similarity(answer_emb, context_emb)[0][0]

            # Method 2: Token Overlap
            # Compare actual words used (lexical overlap)
            answer_tokens = set(answer.lower().split())
            context_tokens = set(context.lower().split())
            # Jaccard-like overlap: what % of answer words appear in context
            overlap = len(answer_tokens & context_tokens) / len(answer_tokens) if answer_tokens else 0

            # Combined Score
            # Average of semantic and lexical similarity
            combined_score = (similarity + overlap) / 2

            # Determine if grounded based on threshold
            return {
                "is_grounded": combined_score > self.similarity_threshold,
                "score": float(combined_score),
                "embedding_similarity": float(similarity),
                "token_overlap": float(overlap),
                "reason": "Grounded" if combined_score > self.similarity_threshold else "Low similarity to context"
            }

        except Exception as e:
            # Handle errors gracefully
            return {
                "is_grounded": False,
                "score": 0.0,
                "reason": f"Verification error: {str(e)[:50]}"
            }

    def detect_hallucination(self, answer: str, context: str) -> Tuple[bool, Dict]:  # Detect if answer contains hallucinated content.
        result = self.verify_grounding(answer, context)
        # Hallucination = NOT grounded in context
        return not result["is_grounded"], result

# Initialize answer verifier
answer_verifier = AnswerVerifier(vector_store, logger)
logger.info("Answer Verifier initialized")

Step 14: Load Llama 2

In [ ]:
# Log the start of Llama 2 loading
logger.info("Loading Llama 2.")

class LLMManager:   # Manages Llama 2 LLM for generation tasks
    # Handles loading and inference with quantized Llama 2 model

    def __init__(self, config: Config, logger: logging.Logger):   # Initialize with configuration and logger.
        self.config = config
        self.logger = logger
        self.llm = None  # Will hold the Llama model instance

    def load_llama(self) -> bool:  # Download (if needed) and load the Llama 2 GGUF model./
        try:
            # Download model from Hugging Face Hub
            # Downloads only if not already cached in local_dir
            model_path = hf_hub_download(
                repo_id=self.config.llm_model_name,      # "TheBloke/Llama-2-7B-GGUF"
                filename=self.config.llm_model_file,     # "llama-2-7b.Q4_K_M.gguf"
                local_dir=self.config.models_dir         # Cache directory
            )

            # Initialize Llama model with optimized settings
            self.llm = Llama(
                model_path=model_path,                   # Path to GGUF file
                n_ctx=4096,                             # Context window size (tokens)
                n_threads=8,                            # CPU threads for inference
                n_gpu_layers=-1 if DEVICE == "cuda" else 0,  # Use GPU if available
                verbose=False,                          # Suppress verbose output
            )

            self.logger.info("Llama 2 loaded")
            return True

        except Exception as e:
            self.logger.error(f"Error loading Llama 2: {e}")
            self.llm = None
            return False

    def generate(self, prompt: str, max_tokens: int = 150) -> str:  # Generate text using Llama 2 with controlled parameters.
        # Return empty if model not loaded
        if self.llm is None:
            return ""

        try:
            # Generate response with conservative settings
            response = self.llm(
                prompt,
                max_tokens=max_tokens,                  # Maximum output length
                temperature=0.3,                        # Low randomness for factual answers
                top_p=0.95,                             # Nucleus sampling threshold
                repeat_penalty=1.1,                     # Penalize repetition
                stop=["\n\n", "Question:", "Context:", "Answer:"],  # Stop tokens
                echo=False,                            # Don't echo prompt in output
            )

            # Extract and clean the generated text
            return response['choices'][0]['text'].strip()

        except Exception as e:
            self.logger.error(f"Error generating with Llama: {e}")
            return ""

# Initialize LLM manager and load model
llm_manager = LLMManager(config, logger)
llm_manager.load_llama()

Step 15: Enhanced answer function

In [ ]:
def clean_and_format_text(text: str) -> str:   # Clean and format generated text for consistency.

    if not text:
        return ""

    # Normalize whitespace (remove extra spaces, newlines)
    text = ' '.join(text.split())

    # Ensure sentence ends with punctuation
    if text and not text[-1] in '.!?':
        text = text + '.'

    # Capitalize first letter
    if text and len(text) > 0:
        text = text[0].upper() + text[1:]

    return text.strip()

def generate_flant5(question: str, max_length: int = 150) -> Optional[str]:   # Generate answer using fine-tuned FLAN-T5 model.

    try:
        # Retrieve relevant context (top 1 chunk)
        results = vector_store.search(question, top_k=1)
        context = results[0]["content"][:400] if results else ""

        # Format prompt with context if available
        if context:
            text = f"Context: {context}\nQuestion: {question}\nAnswer:"
        else:
            text = f"Question: {question}\nAnswer:"

        # Tokenize input
        inputs = flant5_tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

        # Generate with conservative settings
        with torch.no_grad():
            outputs = flant5_model.generate(
                **inputs,
                max_length=max_length,
                temperature=0.2,      # Low temperature for factual answers
                num_beams=2,          # Beam search for better quality
            )

        # Decode and clean
        answer = flant5_tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Filter out very short/incomplete answers
        if len(answer.split()) < 3:
            return None

        return clean_and_format_text(answer)

    except Exception as e:
        logger.error(f"Error generating FLAN-T5: {e}")
        return None

def answer_question_with_expansion(question: str) -> Dict:   # Main answer function with query expansion and verification
    # Complete RAG pipeline: expansion → retrieval → generation → verification

    try:
        # Step 1: Query Expansion
        # Generate multiple query variants for better retrieval
        expanded_queries = query_expander.expand_query(question)

        # Step 2: Retrieve with All Query Variants
        all_results = []
        for q in expanded_queries:
            results = vector_store.search(q, top_k=config.top_k_retrieval)
            all_results.extend(results)

        # Deduplicate results (same content might appear from different queries)
        seen = set()
        unique_results = []
        for r in all_results:
            # Use first 100 chars as dedup key
            if r["content"][:100] not in seen:
                seen.add(r["content"][:100])
                unique_results.append(r)

        # Sort by relevance score (highest first)
        unique_results.sort(key=lambda x: x["score"], reverse=True)
        top_results = unique_results[:config.top_k_retrieval]

        # Handle no results
        if not top_results:
            return {'answer': "I couldn't find information about that.", 'model_used': "No Results"}

        # Extract best context and metadata
        context = top_results[0]["content"][:600]
        chapter = top_results[0]["chapter"]
        title = top_results[0]["title"]

        # Step 3: Generate Answer (Try Llama first, fallback to FLAN-T5)
        # Format prompt for Llama 2 (instruction-style)
        llama_prompt = f"""Answer based ONLY on this textbook excerpt:
{context}

Question: {question}
Answer (one sentence):"""

        # Try Llama 2 first (better quality)
        llama_answer = llm_manager.generate(llama_prompt, max_tokens=80)

        # Fallback to FLAN-T5 if Llama fails or gives short answer
        if llama_answer and len(llama_answer) > 15:
            final_answer = clean_and_format_text(llama_answer)
            model_used = "Llama 2"
        else:
            flant5_answer = generate_flant5(question)
            if flant5_answer:
                final_answer = flant5_answer
                model_used = "FLAN-T5 LoRA"
            else:
                # Last resort: extract directly from context
                final_answer = clean_and_format_text(context[:200] + "...")
                model_used = "Context Extract"

        # Step 4: Answer Verification (Detect Hallucinations)
        verification = answer_verifier.verify_grounding(final_answer, context)

        # Add warning if not grounded in context
        if not verification["is_grounded"]:
            final_answer += " (Note: This answer may not be fully grounded in the textbook context.)"

        # Add source citation and model badge
        citation = f"\n\n Source: Chapter {chapter} - {title}"
        model_badge = f"\n Model: {model_used}"

        # Return comprehensive result
        return {
            'answer': f"{final_answer}{citation}{model_badge}",
            'model_used': model_used,
            'verification_score': verification["score"],
            'is_grounded': verification["is_grounded"]
        }

    except Exception as e:
        logger.error(f"Error answering question: {e}")
        return {'answer': f"Error: {str(e)[:100]}. Please try again.", 'model_used': "Error"}

Step 16: Comprehensive RAG Evaluation

In [ ]:
# Log the start of RAG evaluation
logger.info("Running Comprehensive RAG Evaluation...")

# Test Dataset Creation
def create_test_dataset(chapter_texts: Dict[int, str], size: int = 50) -> pd.DataFrame:
    # Create evaluation dataset from textbook content.
    # Combines predefined concept questions and random sentence-based questions.

    test_questions = []
    test_answers = []
    test_chapters = []

    # Step 1: Extract All Sentences
    # Build pool of sentences for random sampling
    all_sentences = []
    for ch_num, text in chapter_texts.items():
        # Split text into sentences
        sentences = re.split(r'[.!?]+', text)
        for sent in sentences:
            # Only keep sentences with meaningful length
            if len(sent.strip()) > 30:
                all_sentences.append({
                    "text": sent.strip() + ".",  # Add back punctuation
                    "chapter": ch_num
                })

    # Step 2: Create Concept-based Questions
    import random
    random.seed(42)  # For reproducibility

    # Predefined concept→question mapping for key topics
    concept_questions = {
        'force': 'What is force?',
        'friction': 'What is friction?',
        'sound': 'What is sound?',
        'light': 'What is light?',
        'pressure': 'What is pressure?',
        'combustion': 'What is combustion?',
        'microorganisms': 'What are microorganisms?',
        'reproduction': 'What is reproduction?',
        'conservation': 'What is conservation?',
        'adolescence': 'What is adolescence?',
    }

    # Find answers for each concept question
    for concept, question in concept_questions.items():
        answer = ""
        for ch_num, text in chapter_texts.items():
            if concept in text.lower():
                # Find first sentence containing the concept
                sentences = re.split(r'[.!?]+', text)
                for sent in sentences:
                    if concept in sent.lower() and len(sent.strip()) > 20:
                        answer = sent.strip() + "."
                        break
                if answer:
                    break  # Stop once we find an answer

        # Add to dataset if answer found
        if answer:
            test_questions.append(question)
            test_answers.append(answer)
            test_chapters.append(1)  # Default chapter (concepts appear early)

    # Step 3: Add Random Sentence-based Questions
    # Sample random sentences to reach desired test size
    remaining_size = size - len(test_questions)
    sample_sentences = random.sample(all_sentences, min(remaining_size, len(all_sentences)))

    # Create questions by quoting the sentence start
    for item in sample_sentences:
        # Use first 50 chars as question context
        question = f"What is explained in: '{item['text'][:50]}...'?"
        test_questions.append(question)
        test_answers.append(item['text'])
        test_chapters.append(item['chapter'])

    # Step 4: Trim to Requested Size
    test_questions = test_questions[:size]
    test_answers = test_answers[:size]
    test_chapters = test_chapters[:size]

    # Return as DataFrame for easy analysis
    return pd.DataFrame({
        'question': test_questions,
        'reference_answer': test_answers,
        'chapter': test_chapters
    })

# Create and Save Test Dataset
df_test = create_test_dataset(chapter_texts, config.eval_size)

# Save to CSV for later use and analysis
df_test.to_csv(f"{config.eval_dir}/enhanced_test_questions.csv", index=False)

# Log completion
logger.info(f"Created enhanced test dataset with {len(df_test)} questions")

Step 17: Comprehensive Evaluation

In [ ]:
def evaluate_rag_comprehensive(df_test, answer_function, vector_store):  # Comprehensive RAG evaluation with retrieval and generation metrics.
    # Evaluates both retrieval quality and answer quality.

    # Initialize Metrics
    # ROUGE for text similarity (n-gram overlap)
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    # BLEU smoothing for translation quality
    smooth_fn = SmoothingFunction().method4

    results = []

    # Evaluate Each Question
    for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluating"):
        question = row['question']
        reference = row['reference_answer']
        chapter = row['chapter']

        # Step 1: Retrieve Relevant Documents
        if vector_store.index is not None:
            retrieval_results = vector_store.search(question, top_k=config.top_k_retrieval)

            # Count retrieved documents from the correct chapter
            relevant_retrieved = 0
            for r in retrieval_results:
                if r["chapter"] == chapter:
                    relevant_retrieved += 1

            # Total relevant documents in the entire index
            total_relevant = len([d for d in vector_store.mapping if d["chapter"] == chapter])

            # Retrieval Metrics
            # Recall@k: Proportion of relevant documents retrieved
            recall_k = relevant_retrieved / total_relevant if total_relevant > 0 else 0

            # Precision@k: Proportion of retrieved documents that are relevant
            precision_k = relevant_retrieved / len(retrieval_results) if retrieval_results else 0

            # MRR: Mean Reciprocal Rank (position of first relevant result)
            mrr = 0
            for i, r in enumerate(retrieval_results, 1):
                if r["chapter"] == chapter:
                    mrr = 1 / i
                    break
        else:
            recall_k = precision_k = mrr = 0  # No index available

        # Step 2: Generate Answer
        try:
            response = answer_function(question)
            generated = response.get('answer', '')
            model_used = response.get('model_used', 'Unknown')
            is_grounded = response.get('is_grounded', False)
            verification_score = response.get('verification_score', 0)
        except Exception as e:
            generated = f"Error: {str(e)}"
            model_used = 'Error'
            is_grounded = False
            verification_score = 0

        # Step 3: Calculate Generation Metrics

        # BLEU Score: N-gram precision against reference
        try:
            ref_tokens = reference.lower().split()
            gen_tokens = generated.lower().split()
            bleu = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smooth_fn)
        except:
            bleu = 0.0

        # ROUGE Scores: Recall-oriented n-gram overlap
        try:
            rouge_scores = rouge_scorer_obj.score(reference, generated)
            rouge1 = rouge_scores['rouge1'].fmeasure  # Unigram overlap
            rouge2 = rouge_scores['rouge2'].fmeasure  # Bigram overlap
            rougeL = rouge_scores['rougeL'].fmeasure  # Longest common subsequence
        except:
            rouge1 = rouge2 = rougeL = 0.0

        # Step 4: Store Results
        results.append({
            'question_id': idx + 1,
            'chapter': chapter,
            'question': question[:60] + '...' if len(question) > 60 else question,
            # Generation metrics
            'bleu': round(float(bleu), 4),
            'rouge1': round(float(rouge1), 4),
            'rouge2': round(float(rouge2), 4),
            'rougeL': round(float(rougeL), 4),
            # Retrieval metrics
            'precision@k': round(float(precision_k), 4),
            'recall@k': round(float(recall_k), 4),
            'mrr': round(float(mrr), 4),
            # Verification
            'is_grounded': is_grounded,
            'verification_score': round(float(verification_score), 4),
            'model': model_used
        })

    # Save Results
    df_results = pd.DataFrame(results)
    df_results.to_csv(f"{config.eval_dir}/comprehensive_evaluation_results.csv", index=False)

    return df_results

# Run evaluation
df_results = evaluate_rag_comprehensive(df_test, answer_question_with_expansion, vector_store)

Step 18: Evaluation Summary

In [ ]:
# Log Header
logger.info("Comprehensive Evaluation Summary")

# Overall Generation Metrics
logger.info(f"\n Overall Metrics:")
logger.info(f"   • Questions: {len(df_results)}")
logger.info(f"   • Average BLEU: {df_results['bleu'].mean():.4f}")        # Translation quality
logger.info(f"   • Average ROUGE-1: {df_results['rouge1'].mean():.4f}")   # Unigram overlap
logger.info(f"   • Average ROUGE-2: {df_results['rouge2'].mean():.4f}")   # Bigram overlap
logger.info(f"   • Average ROUGE-L: {df_results['rougeL'].mean():.4f}")   # Longest common subsequence

# Retrieval Performance
logger.info(f"\n Retrieval Metrics:")
logger.info(f"   • Average Precision@K: {df_results['precision@k'].mean():.4f}")  # Accuracy of retrieval
logger.info(f"   • Average Recall@K: {df_results['recall@k'].mean():.4f}")      # Completeness of retrieval
logger.info(f"   • Average MRR: {df_results['mrr'].mean():.4f}")                # Ranking quality

# Hallucination Detection
logger.info(f"\n Answer Verification:")
grounded_count = df_results['is_grounded'].sum()  # Count of grounded answers
logger.info(f"   • Grounded Answers: {grounded_count}/{len(df_results)} ({grounded_count/len(df_results)*100:.1f}%)")
logger.info(f"   • Average Verification Score: {df_results['verification_score'].mean():.4f}")

# Chapter-wise Performance
# Analyze which chapters perform best/worst
logger.info(f"\n Performance by Chapter:")
chapter_stats = df_results.groupby('chapter').agg({
    'bleu': 'mean',
    'rougeL': 'mean',
    'precision@k': 'mean',
    'recall@k': 'mean'
}).round(4)
logger.info(chapter_stats.to_string())

# Model-wise Performance
# Compare different models (Llama 2 vs FLAN-T5 vs Context Extract)
logger.info(f"\n Performance by Model:")
model_stats = df_results.groupby('model').agg({
    'bleu': 'mean',
    'rougeL': 'mean',
    'is_grounded': 'mean'  # What % of answers are grounded
}).round(4)
logger.info(model_stats.to_string())

# Save Summary to JSON
# Create comprehensive summary dictionary
summary = {
    'timestamp': datetime.now().isoformat(),                              # When evaluation ran
    'total_questions': len(df_results),
    # Generation metrics
    'avg_bleu': float(df_results['bleu'].mean()),
    'avg_rouge1': float(df_results['rouge1'].mean()),
    'avg_rouge2': float(df_results['rouge2'].mean()),
    'avg_rougeL': float(df_results['rougeL'].mean()),
    # Retrieval metrics
    'avg_precision_k': float(df_results['precision@k'].mean()),
    'avg_recall_k': float(df_results['recall@k'].mean()),
    'avg_mrr': float(df_results['mrr'].mean()),
    # Verification metrics
    'grounded_percentage': float(grounded_count / len(df_results) * 100),
    'avg_verification_score': float(df_results['verification_score'].mean()),
}

# Save summary to JSON for easy parsing
with open(f"{config.eval_dir}/comprehensive_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Log completion
logger.info(f"\n Results saved to {config.eval_dir}")

Step 19: Health Check

In [ ]:
# Log the start of health check
logger.info("Running health check.")

def health_check():  # Verify all system components are properly initialized and ready.
    # Returns True if all checks pass, False otherwise.

    # Define All Health Checks
    checks = {
        "FAISS Index": vector_store.index is not None,              # Vector index loaded
        "Documents": len(vector_store.mapping) > 0,                 # Documents in index
        "Embedder": vector_store.embedder is not None,              # Embedding model loaded
        "Training Data": len(training_pairs) >= 50,                 # Enough training data
        "FLAN-T5 Model": flant5_model is not None,                 # LoRA model loaded
        "Evaluation Results": len(df_results) > 0,                 # Evaluation completed
    }

    # Check if all components are healthy
    all_ok = all(checks.values())

    # Log Results
    logger.info("\n Health Check Results:")
    for check, status in checks.items():
        # Show green checkmark for passing, red X for failing
        logger.info(f"   {check}: {'✅' if status else '❌'}")

    # Return overall health status
    return all_ok

# Run the health check
health_check()

Step 20: Export and Download

In [ ]:
# Log the start of deployment package creation
logger.info("Creating download package.")

# Create Dockerfile
# Dockerfile defines the container environment
dockerfile_content = """
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

RUN mkdir -p ./vector_store ./corpus ./logs ./eval_results

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

# Create requirements.txt
# Python dependencies for the container
requirements = """
llama-cpp-python
haystack-ai
sentence-transformers
faiss-cpu
pymupdf
rouge-score
nltk
pandas
numpy
tqdm
scikit-learn
torch
huggingface_hub
peft
accelerate
transformers
requests
beautifulsoup4
streamlit
pyyaml
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

# Create docker-compose.yml
# Orchestrates multi-container deployment (if needed)
docker_compose = """
version: '3.8'

services:
  rag-system:
    build: .
    ports:
      - "8501:8501"
    volumes:
      - ./vector_store:/app/vector_store
      - ./corpus:/app/corpus
      - ./logs:/app/logs
      - ./lora_adapter:/app/lora_adapter
    environment:
      - STREAMLIT_SERVER_PORT=8501
      - STREAMLIT_SERVER_ADDRESS=0.0.0.0
    restart: unless-stopped
"""

with open("docker-compose.yml", "w") as f:
    f.write(docker_compose)

logger.info("Deployment files created")

# Create and Download ZIP
def create_and_download_zip(folder_paths, zip_name="class8_science_outputs.zip"):   # Package all outputs into a ZIP file and download from Colab.
   # Includes: models, vector store, evaluation results, and deployment files.

    # Step 1: Create ZIP
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        # Add each folder and its contents
        for folder in folder_paths:
            if os.path.exists(folder):
                for root, dirs, files in os.walk(folder):
                    for file in files:
                        file_path = os.path.join(root, file)
                        # Keep relative paths in the ZIP
                        arcname = os.path.relpath(file_path, start='.')
                        zipf.write(file_path, arcname)

        # Add deployment configuration files
        for file in ["Dockerfile", "requirements.txt", "docker-compose.yml"]:
            if os.path.exists(file):
                zipf.write(file, file)

    # Step 2: Download from Colab
    from google.colab import files as colab_files
    colab_files.download(zip_name)

# Define what to include in ZIP
folders_to_zip = [
    "./corpus",          # Document chunks (JSON/JSONL)
    "./vector_store",    # FAISS index and mapping
    "./lora_adapter",    # Trained LoRA adapter weights
    "./eval_results",    # Evaluation metrics and summaries
    "./logs"             # Application logs
]

# Create and download the package
create_and_download_zip(folders_to_zip)

Step 21: Final Status

In [ ]:
# Log Final Production Summary
logger.info("Enhanced System Ready For Production!")

# List All Improvements
logger.info("\n Improvements Implemented:")
logger.info("100+ Training QA Pairs (Auto-generated)")      # Synthetic data generation
logger.info("Enhanced LoRA (r=16, more modules)")           # Better fine-tuning
logger.info("50+ Test Questions")                           # Comprehensive evaluation
logger.info("Query Expansion (Synonyms + Rephrasing)")      # Improved retrieval recall
logger.info("Answer Verification (Grounding Check)")        # Hallucination detection
logger.info("Retrieval Metrics (Precision@K, Recall@K, MRR)") # Retrieval quality
logger.info("BLEU + ROUGE Scores")                          # Generation quality
logger.info("Docker Support")                               # Containerized deployment
logger.info("Comprehensive Logging")                        # Debugging & monitoring

# List All Metrics
logger.info("\n Evaluation Metrics Captured:")
logger.info("   • Generation: BLEU, ROUGE-1, ROUGE-2, ROUGE-L")   # Text quality metrics
logger.info("   • Retrieval: Precision@K, Recall@K, MRR")        # Search quality metrics
logger.info("   • Quality: Grounding Score, Verification Score") # Hallucination metrics

# Deployment Instructions
logger.info("\n Deploy with:")
logger.info("   docker-compose up -d")                           # One-command deployment

# Final Completion Message
logger.info("Production-ready system complete!")

Step 22: Quick Status Check

In [ ]:
# Header
print("Project Status Check")


# 1. File Existence Check
# Verify all critical system files are present
required_files = [
    "./vector_store/faiss_index.bin",                              # FAISS vector index
    "./vector_store/index_mapping.json",                           # Document metadata
    "./corpus/class8_science.jsonl",                               # Document chunks
    "./lora_adapter/final_model/adapter_config.json",              # LoRA config
    "./eval_results/comprehensive_evaluation_results.csv",         # Evaluation data
    "./eval_results/comprehensive_summary.json",                   # Summary metrics
    "./logs/system.log",                                          # Application logs
]

print("\n File Status:")
for file in required_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # Convert bytes to KB
        print(f"   ✅ {file} ({size:.1f} KB)")
    else:
        print(f"   ❌ {file} (missing)")

# 2. Evaluation Results Analysis
# Load and display performance metrics from CSV
if os.path.exists("./eval_results/comprehensive_evaluation_results.csv"):
    df = pd.read_csv("./eval_results/comprehensive_evaluation_results.csv")

    print(f"\n Evaluation Results:")
    print(f"   • Total Questions: {len(df)}")

    # Generation quality metrics
    print(f"   • Avg BLEU: {df['bleu'].mean():.4f}")              # Translation quality
    print(f"   • Avg ROUGE-1: {df['rouge1'].mean():.4f}")         # Unigram overlap
    print(f"   • Avg ROUGE-L: {df['rougeL'].mean():.4f}")         # Longest common subsequence

    # Retrieval quality metrics
    print(f"   • Avg Precision@K: {df['precision@k'].mean():.4f}") # Accuracy of retrieval
    print(f"   • Avg Recall@K: {df['recall@k'].mean():.4f}")      # Completeness of retrieval

    # Grounding/hallucination metrics
    grounded = df['is_grounded'].sum()
    print(f"   • Grounded Answers: {grounded}/{len(df)} ({grounded/len(df)*100:.1f}%)")

# 3. Training Metadata
# Display LoRA training configuration
if os.path.exists("./lora_adapter/training_metadata.json"):
    with open("./lora_adapter/training_metadata.json", "r") as f:
        meta = json.load(f)

    print(f"\n Training Metadata:")
    print(f"   • Training Samples: {meta.get('train_size', 'N/A')}")
    print(f"   • Epochs: {meta.get('epochs', 'N/A')}")
    print(f"   • LoRA r: {meta.get('lora_r', 'N/A')}")
    print(f"   • Learning Rate: {meta.get('learning_rate', 'N/A')}")
    print(f"   • Trainable Params: {meta.get('trainable_params', 'N/A')}")

# 4. Vector Store Statistics
# Check document coverage in FAISS index
if os.path.exists("./vector_store/index_mapping.json"):
    with open("./vector_store/index_mapping.json", "r") as f:
        mapping = json.load(f)

    print(f"\n Vector Store:")
    print(f"   • Documents: {len(mapping)}")

    # Count unique chapters covered
    chapters = set([doc['chapter'] for doc in mapping])
    print(f"   • Chapters covered: {len(chapters)}")

# 5. Error Log Analysis
# Check for errors and warnings in logs
if os.path.exists("./logs/system.log"):
    with open("./logs/system.log", "r") as f:
        log_content = f.read()
        errors = log_content.count("ERROR")      # Count error messages
        warnings = log_content.count("WARNING")  # Count warning messages

    print(f"\n Logs:")
    print(f"   • Errors: {errors}")
    print(f"   • Warnings: {warnings}")
    if errors == 0:
        print("No errors found!")

# Footer
print("Status check complete!")